# Introduction to Machine Learning -- MC886
# Assignment 2 - Model Selection and Regularization

**Professor:** Marcelo da Silva Reis  
**PED:** Heigon Soldera

In [ ]:
# Please, enter your name and RA number in the lines below:
print("Lucas Treviso Bandeira", "|", "322979")
print("Mariana Dourado Ximenes de Sena Santos", "|", "300643")

---

## Objective

This assignment investigates the effects of overfitting in different data regimes
and test conditions.
The goal is to understand the different results of regularization when data
availability changes, and how they impact the generalization across **three**
different distributions: the one used for training, the clean one, and a
corruption the model has never seen.

### Guidelines

> Please read these guidelines carefully.

1. Questions can be answered in English or in Portuguese.

2. Use data visualization as a primary tool for understanding the problem. Inspect the training data, corrupted inputs, and model behavior. When an assumption is uncertain, verify it empirically through plots.

3. The provided code serves only as a reference. Modifications are allowed, including changes to functions and the introduction of new components. Code clarity and organization are required.

4. The use of generative AI tools (e.g., ChatGPT, GitHub Copilot, DeepSeek, LLaMA) is permitted strictly for implementation support. The use of AI for interpreting, analyzing, or discussing experimental results is discouraged. Submissions must include:
   - A declaration of which models were used.
   - The prompts that generated any incorporated code.

5. Do not hesitate to reach out to the PED/PAD via Discord.

### About the Dataset

This assignment uses the [MNIST](https://web.archive.org/web/20240213122836/http://yann.lecun.com/exdb/mnist/) dataset, a standard benchmark for image classification. It consists of 28×28 grayscale images of handwritten digits (0–9), where each pixel encodes an intensity value in [0, 255].

MNIST is useful here because it is simple enough that a linear model can do well on it, which means the learned weights are directly interpretable as images, where each weight corresponds to the influence of each pixel.

To study robustness and distribution shift, the images were corrupted with one of the corruptions of [MNIST-C](https://github.com/google-research/mnist-c) (Mu & Gilmer, 2019). Which corruption it is, is for you to find out in Part 1.

The training set consists entirely of corrupted images. The test set contains **three splits** of the same 10,000 test images:
- `test_corrupted`: corrupted test images (same corruption as training).
- `test_clean`: the original, uncorrupted MNIST test images.
- `test_impulse`: the test images with a **different** corruption, impulse (salt-and-pepper) noise, never seen during training.

Performance on `test_corrupted` measures how well the model learned the training task.
Performance on `test_clean` measures how well the model learned digit recognition.
Performance on `test_impulse` measures whether the robustness acquired during training transfers to a corruption the model was never exposed to.

Your job is to train three different models, each one with different number of
available samples:
- Full-data (60000 samples)
- Partial-data (10000 samples)
- Scarce-data (1000 samples)

Finding the best combination of hyper-parameter for each one of them.

In [ ]:
from typing import Any

from numpy.typing import NDArray
import numpy as np
import pandas as pd
import random

import torch
import torch.nn as nn
from torch import optim

import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility.
# Do not tune this number
# Re-run the notebook from the beginning before submitting.
seed = 42
torch.random.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:
# The dataset is available in the classroom assignment.
# Ensure you have downloaded it and placed it alongside this notebook.

# DO NOT change this cell

def subsample_dataset(
    X: NDArray[Any], y: NDArray[Any], n: int
) -> tuple[NDArray[Any], NDArray[Any]]:
    n_samples = len(y)

    idxs = np.random.choice(
        n_samples, size=n, replace=False
    )

    return X[idxs], y[idxs]

DATA_DIR = "data_2s2026"

X_full = np.load(f"{DATA_DIR}/train/train_images.npy")
y_full = np.load(f"{DATA_DIR}/train/train_labels.npy")

X_partial, y_partial = subsample_dataset(X_full, y_full, 10_000)
X_scarce, y_scarce = subsample_dataset(X_full, y_full, 1_000)

print(f"Full-data set: {X_full.shape[0]} samples")
print(f"Partial-data set: {X_partial.shape[0]} samples")
print(f"Scarce-data set: {X_scarce.shape[0]} samples")

In [ ]:
n_cols = 5
n_rows = 2

fig, axs = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(2*n_cols, 2*n_rows))

fig.suptitle("Corrupted MNIST")

for i in range(n_rows):
    for j in range(n_cols):
        ax = axs[i, j]
        idx = random.randint(0, len(X_full)-1)

        ax.set_title(f"Sample {idx}, target {y_full[idx]}", fontsize=8)
        ax.imshow(X_full[idx], cmap='grey')
        ax.set_axis_off()

---

## Part 1. Data Analysis and Preprocessing

### 1.1 Exploring the corruption

**(1.0 point)**

Before any modeling, understand what the corruption actually does to the data.

Is the corruption random per-pixel noise, or is it a structured perturbation?
Explore the **training** dataset, understand the distribution and the effects of the introduced
noise. Useful tools are per-class mean images, per-pixel statistics (e.g., how often each
pixel is saturated at 255), and looking at many individual samples side by side.

In [ ]:
data_regimes = {
    "Scarce": (X_scarce, y_scarce),
    "Partial": (X_partial, y_partial),
    "Full": (X_full, y_full),
}

fig, axs = plt.subplots(len(data_regimes), 10, figsize=(4*len(data_regimes), 4))
for digit in range(10):
    axs[0, digit].set_title(f"digit {digit}", fontsize=8)

for i, name in enumerate(data_regimes):
    X, y = data_regimes[name]


    for digit in range(10):
        mask = (y == digit)
        mean_img = X[mask].mean(axis=0)
        axs[i, digit].imshow(mean_img, cmap="gray", vmin=0, vmax=255)
        axs[i, digit].set_xticks([])
        axs[i, digit].set_yticks([])
        for spine in axs[i, digit].spines.values():
            spine.set_visible(False)

    axs[i, 0].set_ylabel(
        f"{name}\n({len(X):,} samples)",
        fontsize=8, rotation=0, ha="right", va="center", labelpad=8,
    )

plt.suptitle("Per-class mean images in corrupted training data", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Many samples side by side, a few per class, to see the shape of the corruption.
# A local generator is used so the global random state is not consumed here

n_per_class = 5
rng = np.random.default_rng(seed)

fig, axs = plt.subplots(n_per_class, 10, figsize=(15, 1.6 * n_per_class))

for digit in range(10):
    idxs = rng.choice(np.where(y_full == digit)[0], size=n_per_class, replace=False)
    axs[0, digit].set_title(f"digit {digit}", fontsize=9)

    for row, idx in enumerate(idxs):
        axs[row, digit].imshow(X_full[idx], cmap="gray", vmin=0, vmax=255)
        axs[row, digit].axis("off")

plt.suptitle("Random training samples per class", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Per-pixel statistics in each regime: how often each pixel is saturated (255) and
# how often it is lit at all (> 0). Border activity can help inspect the corruption,
# but digit strokes can also reach this region. Saturated pixels can belong to either
# the digit or the corruption, so these maps do not isolate the noise

pixel_stats = {
    "Saturated (= 255)": lambda images: (images == 255).mean(axis=0),
    "Lit (> 0)": lambda images: (images > 0).mean(axis=0),
}

fig, axs = plt.subplots(len(pixel_stats), len(data_regimes), figsize=(13, 8))

for i, (stat_name, stat_fn) in enumerate(pixel_stats.items()):
    maps = [stat_fn(X[..., 0]) for X, _ in data_regimes.values()]
    vmax = max(m.max() for m in maps)

    for j, (name, stat_map) in enumerate(zip(data_regimes, maps)):
        im = axs[i, j].imshow(stat_map, cmap="viridis", vmin=0, vmax=vmax)
        axs[i, j].set_title(f"{name}-data: {stat_name}", fontsize=10)
        axs[i, j].axis("off")

    fig.colorbar(im, ax=axs[i, :].tolist(), shrink=0.8, label="Fraction of images")

plt.suptitle("Per-pixel frequency in the corrupted training data", fontsize=13)
plt.show()

In [ ]:
# Per-class saturation maps on the Full-data regime, to explore spatial patterns
# across classes. Digit strokes can also saturate, so differences between these maps
# alone do not establish class-dependent corruption. The color scale is clipped so
# the faint background is visible

images_full = X_full[..., 0]

fig, axs = plt.subplots(1, 10, figsize=(16, 2.2))

for digit in range(10):
    sat_map = (images_full[y_full == digit] == 255).mean(axis=0)
    im = axs[digit].imshow(sat_map, cmap="viridis", vmin=0, vmax=0.15)
    axs[digit].set_title(f"digit {digit}", fontsize=9)
    axs[digit].axis("off")

fig.colorbar(im, ax=axs.tolist(), shrink=0.9, label="Fraction saturated")
plt.suptitle("Per-class frequency of saturated pixels (Full-data)", fontsize=13, y=1.08)
plt.show()

In [ ]:
# Inspect intensities and pixel counts in the outer 3-pixel band, per class.
# This region can contain both corruption and digit strokes, so lit-pixel counts
# do not measure corruption alone. Saturation is also not an exclusive marker of
# corruption: the digit itself can contain pixels equal to 255

BORDER = 3
border_mask = np.zeros((28, 28), dtype=bool)
border_mask[:BORDER] = border_mask[-BORDER:] = True
border_mask[:, :BORDER] = border_mask[:, -BORDER:] = True

border_pixels = images_full[:, border_mask]
lit_border = (border_pixels > 0).sum(axis=1)

fig, axs = plt.subplots(1, 3, figsize=(17, 4.5), constrained_layout=True)

# Plot 1: intensity of every lit pixel in the image vs. only in the border band
axs[0].hist(images_full[images_full > 0], bins=64, alpha=0.6, density=True, label="Whole image")
axs[0].hist(border_pixels[border_pixels > 0], bins=64, alpha=0.6, density=True, label="Border band only")
axs[0].set_yscale("log")
axs[0].set_xlabel("Pixel intensity (lit pixels only)")
axs[0].set_ylabel("Density (log scale)")
axs[0].set_title("Intensity of lit pixels")
axs[0].grid(alpha=0.3)
axs[0].legend(frameon=False)

# Plot 2: number of lit pixels in the border band, per class (digit + corruption)
axs[1].boxplot([lit_border[y_full == digit] for digit in range(10)], showmeans=True)
axs[1].set_xticks(range(1, 11))
axs[1].set_xticklabels(range(10))
axs[1].set_xlabel("Digit")
axs[1].set_ylabel("Lit pixels in the border band")
axs[1].set_title("Lit pixels in the border band, per class")
axs[1].grid(axis="y", alpha=0.3)

# Plot 3: number of saturated pixels per image, per class (digit + corruption)
saturated_per_image = (images_full == 255).sum(axis=(1, 2))
axs[2].boxplot([saturated_per_image[y_full == digit] for digit in range(10)], showmeans=True, showfliers=False)
axs[2].set_xticks(range(1, 11))
axs[2].set_xticklabels(range(10))
axs[2].set_xlabel("Digit")
axs[2].set_ylabel("Saturated pixels per image")
axs[2].set_title("Saturated pixels per image, per class (no outliers)")
axs[2].grid(axis="y", alpha=0.3)

plt.show()

border_summary = pd.DataFrame({
    "images with lit border (%)": [100 * (lit_border[y_full == d] > 0).mean() for d in range(10)],
    "mean lit border pixels": [lit_border[y_full == d].mean() for d in range(10)],
    "mean saturated pixels": [saturated_per_image[y_full == d].mean() for d in range(10)],
}, index=pd.Index(range(10), name="digit"))

print(f"Images with at least one lit pixel in the border band: {100 * (lit_border > 0).mean():.1f}%")
print(f"Lit border pixels that are exactly 255: {100 * (border_pixels[border_pixels > 0] == 255).mean():.1f}%")
display(border_summary.round(1).T)

> **Question 1.1**
>
> (a) Describe the corruption. Is it random and per-sample, or is it systematic and shared across images? Does it vary across digit classes?
>
> (b) Why does the corruption turn into a faint, nearly uniform background in the per-class mean images? What does that tell you about using mean images to diagnose a corruption?
>
> (c) The corruption is made of strokes, and digits are also made of strokes. Which digits do you expect to be most easily confused with (or by) the corruption?
>
> (d) Are there data regimes where the corruption can affect the model more than in others?
> Hypothesize how these different regimes will affect a model trained on them.

(a) A corrupção caracteriza-se pela adição de traços brancos em zigue-zague, compostos por segmentos conectados e frequentemente formados por pixels saturados. Esses traços podem aparecer no fundo da imagem ou se sobrepor ao dígito, conforme ilustrado no gráfico **“Random training samples per class”**. Embora sua posição e orientação variem entre as amostras, a perturbação apresenta uma estrutura espacial definida, na qual os pixels afetados formam traços conectados, em vez de ruído independente por pixel. Dessa forma, o padrão específico da perturbação varia de uma imagem para outra, mas sua natureza permanece a mesma, correspondendo à categoria zigzag, descrita no artigo **[“MNIST-C: A Robustness Benchmark for Computer Vision”](https://arxiv.org/abs/1906.02337)**. Além disso, não há evidências de variação sistemática da corrupção entre as classes, uma vez que as imagens médias e os mapas de saturação apresentam padrões semelhantes, enquanto a quantidade média de pixels saturados apresenta pequenas diferenças entre os dígitos.

(b) Como os traços da corrupção aparecem em posições e orientações diferentes em cada imagem, seus pixels mais intensos se diluem ao calcular a média de muitas amostras, formando um fundo fraco e aproximadamente uniforme, como observado no gráfico **“Per-class mean images in corrupted training data”**. Já os traços característicos de cada dígito aparecem em regiões semelhantes dentro da mesma classe e, por isso, permanecem visíveis na média. Isso mostra que a média das imagens revela padrões espaciais recorrentes, mas pode ocultar a estrutura de uma corrupção que varia entre amostras. Assim, é necessário examinar imagens individuais, como as do gráfico **“Random training samples per class”**, para perceber que a corrupção é formada por traços.

(c) Esperamos que os dígitos 1, 7 e 4 sejam especialmente suscetíveis à confusão, pois possuem traços retos semelhantes aos introduzidos pela corrupção. No gráfico **“Random training samples per class”**, observamos segmentos brancos que aparecem isolados ou se sobrepõem aos dígitos, podendo ser interpretados como parte deles. Por exemplo, um traço próximo ao topo de um 1 pode fazê-lo se parecer com um 7, enquanto segmentos adicionais podem criar cruzamentos semelhantes aos presentes no 4. O 1 também pode ser confundido com um traço isolado da própria corrupção.

(d) Esperamos que a corrupção afete mais o aprendizado no regime Scarce, com 1.000 amostras, pois há menos exemplos para distinguir os traços característicos dos dígitos de padrões ocasionais da corrupção. Com poucos dados, o modelo pode aprender associações acidentais entre os traços da corrupção e as classes, aumentando o risco de sobreajuste e prejudicando a generalização. No regime Partial, com 10.000 amostras, esperamos um comportamento intermediário; no Full, com 60.000, a maior diversidade de exemplos deve permitir estimar padrões mais estáveis e reduzir essas associações acidentais.

### 1.2 Preprocessing

**(1.0 point)**

Before training, the raw data must be brought into a form suitable for gradient-based optimization.

Two considerations are important here:

1. **Scale.** Raw pixel values are in [0, 255]. Gradient descent is sensitive to the scale of inputs; large input values produce large gradients, which can destabilize training.

2. **Data leakage.** Any statistics used for preprocessing must be computed from the training set only, then applied to all other splits.
Computing statistics on the full dataset, including test data, leaks information about the test distribution into preprocessing and invalidates evaluation.

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Pixel intensities live in the fixed range [0, 255], known before seeing any data,
# so dividing by 255 estimates nothing from the samples.
# Each pixel is then centered on its mean over the training images. This is the only
# statistic learned from data, so it is always fit on the training split (or on the
# training folds, inside cross-validation) and reused unchanged on everything else.
# The scale is not divided by the per-pixel standard deviation: border pixels are
# almost always zero, and standardizing them would turn a rare corruption stroke into
# a huge input value. Keeping every pixel in the same units also keeps the weight
# images comparable across pixels, and makes L1/L2 penalize every pixel equally.

PIXEL_MAX = 255.0

def flatten_and_scale(X: NDArray[np.uint8]) -> NDArray[np.float32]:
    """Flattens (N, 28, 28, 1) images into (N, 784) with values in [0, 1]."""
    return X.reshape(len(X), -1).astype(np.float32) / PIXEL_MAX


class Preprocessor:
    """Scales pixels to [0, 1] and centers them on the mean of the data it was fit on."""

    def fit(self, X: NDArray[np.uint8]) -> "Preprocessor":
        self.pixel_mean = flatten_and_scale(X).mean(axis=0)
        return self

    def transform(self, X: NDArray[np.uint8]) -> torch.Tensor:
        return torch.from_numpy(flatten_and_scale(X) - self.pixel_mean)


def to_labels(y: NDArray[np.int64]) -> torch.Tensor:
    return torch.from_numpy(y).long()


preprocessors = {name: Preprocessor().fit(X) for name, (X, _) in data_regimes.items()}

for name, (X, y) in data_regimes.items():
    X_prep = preprocessors[name].transform(X)
    print(
        f"{name}-data: X {tuple(X_prep.shape)}, values in [{X_prep.min():.2f}, {X_prep.max():.2f}], "
        f"mean {X_prep.mean():.1e}, y {y.shape}"
    )

# Why centering matters for gradient descent: with all inputs >= 0, the mean image is a
# direction of very large curvature of the loss. Its size is the largest eigenvalue of
# X^T X / N, which bounds the learning rate that full-batch gradient descent tolerates
X_scaled = torch.from_numpy(flatten_and_scale(X_full))
X_centered = preprocessors["Full"].transform(X_full)
for label, data in [("scaled only", X_scaled), ("scaled and centered", X_centered)]:
    largest_eigenvalue = torch.linalg.eigvalsh(data.T @ data / len(data))[-1].item()
    print(f"Largest eigenvalue of X^T X / N, {label}: {largest_eigenvalue:.1f}")

del X_scaled, X_centered, X_prep

> **Question 1.2**
>
> Describe the preprocessing choices made above. What strategy have you chosen, and why?

Adotamos o reescalonamento das intensidades seguido da centralização por pixel. Primeiro, cada imagem de 28 × 28 pixels é convertida em um vetor de 784 atributos do tipo float32, e suas intensidades são divididas por 255, passando para o intervalo [0, 1], com o objetivo de reduzir a escala das entradas e favorecer a estabilidade da otimização por gradiente. Em seguida, subtraímos de cada atributo sua média no treinamento, deixando a média de cada pixel aproximadamente zero nesse conjunto. Isso remove o componente médio comum às imagens e pode tornar a otimização por gradiente mais estável. Após a centralização, os valores podem ser negativos e não ficam mais restritos a [0, 1].

Não dividimos os atributos pelo desvio padrão de cada pixel. O gráfico **“Per-pixel frequency in the corrupted training data”** mostra que pixels próximos às bordas são ativados com menor frequência. Dividir por desvios padrão pequenos poderia amplificar essas ativações raras, inclusive as causadas pela corrupção. Mantendo uma escala comum para todos os pixels, também facilitamos a comparação dos pesos e evitamos que a padronização altere a intensidade relativa da regularização entre atributos.

Para evitar vazamento de dados, calculamos a média separadamente para cada regime. Na validação cruzada, essa média é recalculada somente nos folds usados para treinar e aplicada sem alterações ao fold de validação. No modelo final, a média é estimada com todo o treinamento do respectivo regime e reutilizada nos conjuntos de teste.

---

## Part 2. Multinomial Logistic Regression

### 2.1 The model

Multinomial logistic regression (also called softmax regression) is the natural extension of binary logistic regression to $K > 2$ classes.

Given an input $\mathbf{x} \in \mathbb{R}^d$, the model computes a score (logit) for each class $k$:

$$z_k = \mathbf{w}_k^\top \mathbf{x} + b_k$$

Each logit is an unconstrained real number, in order to transform these values into
probabilities, where $p_k \geq 0$ and $\sum_k p_k = 1$, the softmax function
is used:

$$P(y = k \mid \mathbf{x}) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

Training minimizes the **cross-entropy loss**, which is the negative log-likelihood of the true class under this distribution:

$$\mathcal{L}(\mathbf{W}, \mathbf{b}) = -\frac{1}{N} \sum_{i=1}^{N} \log P(y_i \mid \mathbf{x}_i)$$

For MNIST with $d = 784$ pixels and $K = 10$ classes, the weight matrix $\mathbf{W} \in \mathbb{R}^{10 \times 784}$ has one row per class. Each row $\mathbf{w}_k$ can be reshaped into a 28×28 image and visualized directly: positive values indicate pixels whose brightness increases the log-odds of class $k$, whereas negative values suppress it.
This direct interpretability is one reason we use a linear model here.

### 2.2 Regularization

Standard cross-entropy minimization has no preference over the parameter space.
With enough capacity, or with corrupted data that contains spurious features, the model can assign large weights to uninformative or noisy pixels if doing so reduces training loss.

Regularization considers an additional assumption over the nature of the learned weights,
adding a penalty on the weights to the loss:

$$\mathcal{L}_{\text{reg}} = \mathcal{L} + \lambda \cdot \Omega(\mathbf{W})$$

Two common choices for $\Omega$:

- **L2 (Ridge):** $\Omega(\mathbf{W}) = \|\mathbf{W}\|_2^2 = \sum_{k,j} w_{kj}^2$. Penalizes large weights uniformly, where all weights shrink toward zero but none are forced exactly to zero.
In PyTorch's optimizers, L2 is implemented as `weight_decay`.

- **L1 (Lasso):** $\Omega(\mathbf{W}) = \|\mathbf{W}\|_1 = \sum_{k,j} |w_{kj}|$. Penalizes the sum of absolute values, producing sparse solutions where many weights are driven to exactly zero.

The hyperparameter $\lambda \geq 0$ controls the strength of regularization.
$\lambda = 0$ recovers the unregularized problem.

**(2.0 point)**

Implement the Multinomial Logistic Regression model using PyTorch's nn.Module
for the linear model (image as input and logits as outputs), and SGD with both
L1 and L2 regularization methods.

In [ ]:
from typing import NamedTuple

N_FEATURES = 28 * 28
N_CLASSES = 10

# Optimization settings shared by every model in the notebook, so differences between
# models come from the data regime and the regularization only. With centered inputs,
# lr = 1.0 is stable for every lambda of the grid (on the uncentered data it already
# oscillates once a penalty is added), and 500 full-batch iterations are enough for the
# regularized models to converge. The unregularized loss keeps decreasing slowly,
# mainly on the Scarce-data regime, whose 1000 images are linearly separable: there the
# loss only reaches zero with infinitely large weights.
LR = 1.0
N_ITER = 500


class LinearModel(nn.Module):
    """Multinomial logistic regression: one linear score (logit) per class."""

    def __init__(self, n_features: int = N_FEATURES, n_classes: int = N_CLASSES) -> None:
        super().__init__()
        self.linear = nn.Linear(n_features, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Accepts both flattened (N, 784) and image-shaped (N, 28, 28, 1) inputs
        return self.linear(x.flatten(start_dim=1))


def accuracy(logits: torch.Tensor, y: torch.Tensor) -> float:
    """Fraction of correctly predicted samples."""
    y_pred = logits.argmax(dim=-1)
    return (y == y_pred).float().mean().item()


def train(
    model: nn.Module,
    X: torch.Tensor,
    y: torch.Tensor,
    n_iter: int,
    lr: float = 0.1,
    l1: float = 0.0,
    l2: float = 0.0
) -> list[float]:
    """
    Train a model using full-batch SGD with L1/L2 regularization.

    Only the weight matrix is regularized: the bias just sets the prior log-odds of
    each class and has no pixel to be shrunk towards zero.

    Returns the list of per-iteration training losses (cross-entropy term only, so the
    curves of models with different penalties stay comparable).
    """
    weight, bias = model.linear.weight, model.linear.bias

    # L2: weight_decay adds l2 * w to the gradient, which is the gradient of the
    # penalty (l2 / 2) * ||W||^2
    optimizer = optim.SGD(
        [
            {"params": [weight], "weight_decay": l2},
            {"params": [bias], "weight_decay": 0.0},
        ],
        lr=lr,
    )
    loss_fn = nn.CrossEntropyLoss()
    losses = []

    for _ in range(n_iter):
        optimizer.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        optimizer.step()

        # L1: proximal step (soft-thresholding) after the gradient step, instead of
        # adding l1 * ||W||_1 to the loss. The subgradient of |w| is l1 * sign(w), so
        # with the penalty in the loss a weight near zero keeps jumping over it by
        # lr * l1 and never settles at exactly zero. Soft-thresholding is the exact
        # minimizer of the penalty around the gradient step: any weight smaller than
        # lr * l1 in magnitude becomes exactly zero, which is the sparsity L1 promises
        # and what the sparsity measurement of Part 4 relies on.
        if l1 > 0:
            with torch.no_grad():
                weight.copy_(torch.sign(weight) * torch.clamp(weight.abs() - lr * l1, min=0.0))

        losses.append(loss.item())

    return losses


class FittedModel(NamedTuple):
    """A trained model together with the preprocessor fit on its training data."""
    model: LinearModel
    preprocessor: Preprocessor
    losses: list[float]


def fit_model(X: NDArray[np.uint8], y: NDArray[np.int64], l1: float = 0.0, l2: float = 0.0) -> FittedModel:
    """Fits the preprocessor and the model on the given (raw) training data."""
    preprocessor = Preprocessor().fit(X)

    # Every model starts from the same initial weights, so differences between them
    # come from the data and the penalty, not from the initialization
    torch.manual_seed(seed)
    model = LinearModel()

    losses = train(model, preprocessor.transform(X), to_labels(y), n_iter=N_ITER, lr=LR, l1=l1, l2=l2)
    return FittedModel(model, preprocessor, losses)


def predict_logits(fitted: FittedModel, X: NDArray[np.uint8]) -> torch.Tensor:
    with torch.no_grad():
        return fitted.model(fitted.preprocessor.transform(X))


def evaluate(fitted: FittedModel, X: NDArray[np.uint8], y: NDArray[np.int64]) -> float:
    """Accuracy of a fitted model on raw images, preprocessed with its own training statistics."""
    return accuracy(predict_logits(fitted, X), to_labels(y))


def get_weights(fitted: FittedModel) -> NDArray[np.float32]:
    """Weight matrix (10, 784) as a numpy array."""
    return fitted.model.linear.weight.detach().numpy().copy()

### 2.3 Baseline mode

**(2.0 points)**

Train a unregularized model on each training set and observe the training loss
per iteration on each model during training.

To diagnose and understand what the trained model have learned, we visualize the
learned weights as a heatmap.

In [ ]:
# Unregularized baseline on each data regime

baseline_models = {name: fit_model(X, y) for name, (X, y) in data_regimes.items()}

fig, ax = plt.subplots(figsize=(10, 5))

for name, fitted in baseline_models.items():
    ax.plot(fitted.losses, label=f"{name}-data ({len(data_regimes[name][1])} samples)")

ax.set_yscale("log")
ax.set_xlabel("Iteration")
ax.set_ylabel("Training cross-entropy (log scale)")
ax.set_title("Training loss of the unregularized baseline")
ax.grid(alpha=0.3)
ax.legend(frameon=False)
plt.show()

baseline_summary = pd.DataFrame({
    "final train loss": [fitted.losses[-1] for fitted in baseline_models.values()],
    "loss drop in the last 100 iterations": [fitted.losses[-101] - fitted.losses[-1] for fitted in baseline_models.values()],
    "train accuracy": [evaluate(fitted, X, y) for fitted, (X, y) in zip(baseline_models.values(), data_regimes.values())],
}, index=pd.Index(baseline_models.keys(), name="regime"))

display(baseline_summary.round(4))

In [ ]:
# Visualize learned weight images, one per digit class.
# Use a diverging colormap: red = positive weight (pixel increases log-odds),
# blue = negative weight (pixel suppresses log-odds).
# The color scale is symmetric around zero, otherwise the colormap center would not
# match a zero weight and the sign of each pixel could not be read from the color

def plot_weights(weights: NDArray[np.float32], title: str) -> None:
    fig, axs = plt.subplots(2, 5, figsize=(12, 5))
    fig.suptitle(title, fontsize=13)

    limit = np.abs(weights).max()

    for digit in range(10):
        ax = axs[digit // 5, digit % 5]
        im = ax.imshow(weights[digit].reshape(28, 28), cmap="icefire", vmin=-limit, vmax=limit)
        ax.set_title(f"digit {digit}", fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    fig.colorbar(im, ax=axs.ravel().tolist(), shrink=0.8, label="Weight")
    plt.show()


# Numbers behind the weight images: overall size, sparsity, and how much of the weight
# magnitude sits on the border band, where (Part 1) almost only the corruption appears

SPARSITY_THRESHOLD = 1e-3

def sparsity(weights: NDArray[np.float32]) -> float:
    """Fraction of weights with magnitude below the threshold."""
    return (np.abs(weights) < SPARSITY_THRESHOLD).mean()


def weight_summary(weights: NDArray[np.float32]) -> dict[str, float]:
    magnitude = np.abs(weights)
    return {
        "max |w|": magnitude.max(),
        "||W||_2": np.sqrt((weights ** 2).sum()),
        "sparsity": sparsity(weights),
        "border share of |w| (%)": 100 * magnitude[:, border_mask.ravel()].sum() / magnitude.sum(),
    }


print(f"The border band holds {100 * border_mask.mean():.1f}% of the pixels")

In [ ]:
for name, fitted in baseline_models.items():
    plot_weights(get_weights(fitted), f"Unregularized baseline, {name}-data")

display(pd.DataFrame(
    {name: weight_summary(get_weights(fitted)) for name, fitted in baseline_models.items()}
).T.round(3))

> **Question 2.3**
>
> Inspect the weight images for each baseline model. How do you explain the
> weight values in the heatmap, and what patterns do they show?
>
> Compare between the model, how are they different?

Cada mapa de calor representa os pesos associados a uma classe, reorganizados em uma matriz de 28 × 28 pixels. Conforme a expressão dos logits da classe $k$, apresentada na Seção 2.1, pesos positivos indicam que o aumento da intensidade de um pixel contribui para elevar o logit da classe, enquanto pesos negativos produzem o efeito contrário. Na visualização, as cores quentes representam pesos positivos, as cores frias indicam pesos negativos e as regiões escuras correspondem a valores próximos de zero.

Nos três regimes de treinamento, observam-se padrões associados à estrutura dos dígitos, especialmente na região central das imagens. Por exemplo, o mapa da classe 1 apresenta uma faixa vertical de pesos positivos, enquanto o da classe 0 exibe pesos negativos na região central e positivos em partes do contorno. Isso ocorre porque a softmax depende apenas das diferenças entre os logits

A comparação entre os modelos revela diferenças na distribuição espacial dos pesos. O Scarce apresenta padrões mais irregulares, com pesos dispersos pelo fundo e pelas bordas. O Full, por sua vez, apresenta estruturas centrais mais definidas e menor presença de pesos expressivos nessas regiões periféricas, enquanto o Partial apresenta características intermediárias. Essa tendência também é observada nas estatísticas:

- A norma dos pesos diminui progressivamente de 13,653 no Scarce para 11,929 no Partial e 9,798 no Full.
- A participação das bordas na soma das magnitudes dos pesos também diminui, passando de 25,8% no Scarce para 24,3% no Partial e 17,8% no Full.

Esses resultados são compatíveis com a hipótese levantada na Questão 1.1(d): com menos exemplos de treinamento, o modelo pode atribuir importância a associações ocasionais entre pixels afetados pela corrupção e determinadas classes. À medida que a quantidade de dados aumenta, a influência dessas associações tende a diminuir, favorecendo padrões mais consistentes com a estrutura dos dígitos.

Entretanto, os mapas de calor não permitem determinar quais pesos estão diretamente associados à corrupção, uma vez que ela pode se sobrepor aos dígitos e que as regiões periféricas também podem conter traços relevantes para a classificação.

O gráfico “Training loss of the unregularized baseline” mostra comportamentos distintos. No Full, a loss praticamente se estabiliza em 0,3561 (acurácia de treino de 89,5%), caindo apenas 0,0052 nas últimas 100 iterações. No Partial, termina em 0,2560 (92,9%) e ainda está caindo. No Scarce, cai continuamente até 0,0232, com 100% de acurácia de treino, sem sinal de estabilização.

A acurácia de 100% no treinamento do Scarce mostra que o modelo linear conseguiu classificar corretamente todas as 1.000 amostras disponíveis. Isso não significa que a entropia cruzada seja zero, pois mesmo com todas as classificações corretas, a perda pode continuar diminuindo à medida que aumenta a probabilidade atribuída às classes verdadeiras. No caso separável, sem regularização, essa redução pode ocorrer com o crescimento ilimitado dos pesos. Já no Full, o modelo alcança 89,5% de acurácia de treinamento após 500 iterações. Em conjunto com os mapas e as estatísticas dos pesos, esses resultados são compatíveis com um ajuste maior às particularidades da amostra no Scarce, mas não comprovam sobreajuste nem permitem concluir como cada modelo generaliza.

---

## Part 3. Regularization

**(2.0 points)**

We now introduce regularization and sweep over $\lambda$ values for both L1 and L2 penalties.

Your **only decision criterion** in this part is cross-validation accuracy computed on the corrupted training data, exactly what you would have in a real scenario without access to a clean or differently corrupted test set.

For each configuration (data regime x reg type x $\lambda$), use K-fold cross-validation to estimate validation accuracy (report mean **and** standard deviation across folds).
Then, for each regularizer, **select the best $\lambda$** by CV and train a final model on the full training set of that regime.

Also train and store one model for **every** $\lambda$ of the grid on the Full-data regime: they will not be used for selection, only for the diagnostic plots of Part 4.

> **Decision task** Report, per regularizer and data regimes, which $\lambda$ you selected and why.
Inspect the weight images of your chosen models and describe what you observe.

In [ ]:
# Implement K-fold cross-validation with correct pre-processing

import time

K_FOLDS = 5

# Log-spaced grids. Their ranges come from a first coarse sweep on the training data,
# so that each one goes from no visible effect up to a clear drop in CV accuracy
LAMBDA_GRID = {
    "L2": [1e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0],
    "L1": [1e-6, 1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2],
}


def penalty(regularizer: str, lam: float) -> dict[str, float]:
    """Keyword arguments of fit_model for a regularizer ("None", "L1" or "L2") and lambda."""
    return {"None": {}, "L1": {"l1": lam}, "L2": {"l2": lam}}[regularizer]


def fit_models(
    X: NDArray[np.uint8], y: NDArray[np.int64], penalties: list[dict[str, float]]
) -> list[FittedModel]:
    """
    Same as [fit_model(X, y, **p) for p in penalties], with all the models trained at once.

    A full-batch iteration of this small model is limited by reading X from memory, not
    by arithmetic, so the models are stacked side by side into one wide linear layer and
    X is read once per iteration for all of them (about 4x faster on the lambda sweep).
    The stacked models share no parameter and the loss is the sum of their
    cross-entropies, so each block gets exactly the gradient it would get alone; the L2
    and L1 steps of train are applied with one lambda per block.
    """
    preprocessor = Preprocessor().fit(X)
    X_train, y_train = preprocessor.transform(X), to_labels(y)
    n_models = len(penalties)

    # Same initial weights as fit_model, repeated for every block
    torch.manual_seed(seed)
    init = LinearModel().linear
    stacked = nn.Linear(N_FEATURES, n_models * N_CLASSES)
    with torch.no_grad():
        stacked.weight.copy_(init.weight.repeat(n_models, 1))
        stacked.bias.copy_(init.bias.repeat(n_models))

    # One lambda per weight row, so it broadcasts over the N_CLASSES rows of each block
    def lambda_per_row(key: str) -> torch.Tensor:
        return torch.tensor([p.get(key, 0.0) for p in penalties]).repeat_interleave(N_CLASSES)[:, None]

    l1, l2 = lambda_per_row("l1"), lambda_per_row("l2")

    # Flattened logits have one row per (sample, model) pair, sample-major
    y_rows = y_train.repeat_interleave(n_models)
    optimizer = optim.SGD(stacked.parameters(), lr=LR)
    losses = []

    for _ in range(N_ITER):
        optimizer.zero_grad()
        logits = stacked(X_train).view(-1, N_CLASSES)
        model_losses = nn.functional.cross_entropy(logits, y_rows, reduction="none").view(-1, n_models).mean(dim=0)
        model_losses.sum().backward()

        with torch.no_grad():
            weight = stacked.weight
            weight.grad += l2 * weight  # what weight_decay does in train
            optimizer.step()
            weight.copy_(torch.sign(weight) * torch.clamp(weight.abs() - LR * l1, min=0.0))

        losses.append(model_losses.detach())

    losses = torch.stack(losses)
    fitted = []
    for i in range(n_models):
        rows = slice(i * N_CLASSES, (i + 1) * N_CLASSES)
        model = LinearModel()
        with torch.no_grad():
            model.linear.weight.copy_(stacked.weight[rows])
            model.linear.bias.copy_(stacked.bias[rows])
        fitted.append(FittedModel(model, preprocessor, losses[:, i].tolist()))

    return fitted


def cross_validate(
    X: NDArray[np.uint8], y: NDArray[np.int64], penalties: list[dict[str, float]], k: int = K_FOLDS
) -> NDArray[np.float64]:
    """
    Validation accuracy of each penalty (rows) on each of the k folds (columns).

    The folds are stratified, so every fold keeps the class proportions, and fixed by
    the seed, so every configuration is evaluated on exactly the same splits. The raw
    images are split first and fit_models fits the preprocessor on the training folds
    only, so the pixel mean never sees the validation fold.
    """
    folds = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
    scores = []

    for train_idx, val_idx in folds.split(X, y):
        fitted = fit_models(X[train_idx], y[train_idx], penalties)
        scores.append([evaluate(f, X[val_idx], y[val_idx]) for f in fitted])

    return np.array(scores).T


configurations = [("None", 0.0)] + [
    (regularizer, lam) for regularizer, lambdas in LAMBDA_GRID.items() for lam in lambdas
]

# This is the slowest cell of the notebook (around ten minutes on a laptop CPU),
# almost all of it spent on the Full-data folds
cv_rows = []
for regime, (X, y) in data_regimes.items():
    start = time.time()
    all_scores = cross_validate(X, y, [penalty(regularizer, lam) for regularizer, lam in configurations])

    for (regularizer, lam), fold_scores in zip(configurations, all_scores):
        cv_rows.append({
            "regime": regime,
            "regularizer": regularizer,
            "lambda": lam,
            "cv_mean": fold_scores.mean(),
            "cv_std": fold_scores.std(),
            "fold_scores": fold_scores,
        })

    print(f"{regime}-data: {len(configurations)} configurations x {K_FOLDS} folds in {time.time() - start:.0f}s")

cv_results = pd.DataFrame(cv_rows)

In [ ]:
# Mean and standard deviation across folds of every configuration: one column per
# regime and one row per regularizer and lambda, in the order of the grid

cv_text = cv_results["cv_mean"].map("{:.4f}".format) + " ± " + cv_results["cv_std"].map("{:.4f}".format)
cv_table = (
    cv_results.assign(cv=cv_text)
    .pivot(index=["regularizer", "lambda"], columns="regime", values="cv")
    .reindex(index=pd.MultiIndex.from_tuples(configurations), columns=list(data_regimes))
)

# lambda as text, since rounding would show 3e-05 as 0.0000
cv_table.index = pd.MultiIndex.from_tuples(
    [(regularizer, f"{lam:g}") for regularizer, lam in configurations], names=["regularizer", "lambda"]
)
cv_table.columns.name = None

print(f"{K_FOLDS}-fold CV accuracy on the corrupted training data (mean ± std across folds):")
display(cv_table)

In [ ]:
# CV accuracy against lambda, one panel per regime. The shaded bands are +- one
# standard deviation across folds; the dashed line is the unregularized baseline

reg_colors = {"None": "tab:gray", "L2": "tab:blue", "L1": "tab:orange"}

def cv_curve(regime: str, regularizer: str) -> pd.DataFrame:
    rows = (cv_results["regime"] == regime) & (cv_results["regularizer"] == regularizer)
    return cv_results[rows].sort_values("lambda")


# Selection: the lambda with the highest mean CV accuracy, per regime and regularizer
selected_rows = []
for regime in data_regimes:
    for regularizer in LAMBDA_GRID:
        curve = cv_curve(regime, regularizer).reset_index(drop=True)
        best = curve["cv_mean"].idxmax()

        # Paired comparison with the neighbours of the best lambda: every configuration
        # uses the same folds, so the fold-by-fold difference removes the part of the
        # variability that comes from how hard each fold is
        neighbour_gaps = {}
        for side, neighbour in [("left", best - 1), ("right", best + 1)]:
            if 0 <= neighbour < len(curve):
                diff = curve.loc[best, "fold_scores"] - curve.loc[neighbour, "fold_scores"]
                neighbour_gaps[f"gap to {side} neighbour"] = diff.mean()
                neighbour_gaps[f"folds won vs {side}"] = int((diff > 0).sum())
            else:
                neighbour_gaps[f"gap to {side} neighbour"] = np.nan
                neighbour_gaps[f"folds won vs {side}"] = np.nan

        baseline_row = cv_curve(regime, "None").iloc[0]
        selected_rows.append({
            "regime": regime,
            "regularizer": regularizer,
            "lambda": curve.loc[best, "lambda"],
            "cv_mean": curve.loc[best, "cv_mean"],
            "cv_std": curve.loc[best, "cv_std"],
            "gain over baseline": curve.loc[best, "cv_mean"] - baseline_row["cv_mean"],
            **neighbour_gaps,
        })

selected = pd.DataFrame(selected_rows)

fig, axs = plt.subplots(1, len(data_regimes), figsize=(18, 5), constrained_layout=True)

for ax, regime in zip(axs, data_regimes):
    baseline_row = cv_curve(regime, "None").iloc[0]
    ax.axhline(baseline_row["cv_mean"], color=reg_colors["None"], linestyle="--", label="No regularization")
    ax.axhspan(
        baseline_row["cv_mean"] - baseline_row["cv_std"],
        baseline_row["cv_mean"] + baseline_row["cv_std"],
        color=reg_colors["None"], alpha=0.12,
    )

    for regularizer in LAMBDA_GRID:
        curve = cv_curve(regime, regularizer)
        color = reg_colors[regularizer]
        ax.plot(curve["lambda"], curve["cv_mean"], marker="o", color=color, label=regularizer)
        ax.fill_between(
            curve["lambda"], curve["cv_mean"] - curve["cv_std"], curve["cv_mean"] + curve["cv_std"],
            color=color, alpha=0.15,
        )

        choice = selected[(selected["regime"] == regime) & (selected["regularizer"] == regularizer)].iloc[0]
        ax.scatter(choice["lambda"], choice["cv_mean"], marker="*", s=250, color=color,
                   edgecolor="black", zorder=5)

    ax.set_xscale("log")
    ax.set_xlabel(r"$\lambda$")
    ax.set_ylabel("CV accuracy")
    ax.set_title(f"{regime}-data ({len(data_regimes[regime][1])} samples)")
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, loc="lower left")

plt.suptitle(f"{K_FOLDS}-fold CV accuracy on the corrupted training data (stars: selected lambda)", fontsize=13)
plt.show()

print("Selected lambda per regime and regularizer (highest mean CV accuracy):")
# lambda as text, since rounding to 4 decimals would show 3e-05 as 0.0000
display(selected.astype({"lambda": str}).round(4))

In [ ]:
# Diagnostic models for Part 4: one Full-data model for every lambda of the grid.
# They are not used for any selection

grid_configurations = [(regularizer, lam) for regularizer, lambdas in LAMBDA_GRID.items() for lam in lambdas]
full_grid_models = dict(zip(
    grid_configurations,
    fit_models(X_full, y_full, [penalty(regularizer, lam) for regularizer, lam in grid_configurations]),
))

# Final models: the baseline and, per regularizer, the CV-selected lambda retrained on
# the whole training set of each regime. Training is deterministic, so the Full-data
# models are taken from the grid above instead of being trained twice
final_models = {}
for regime, (X, y) in data_regimes.items():
    final_models[(regime, "None")] = baseline_models[regime]

    for regularizer in LAMBDA_GRID:
        lam = selected.query("regime == @regime and regularizer == @regularizer")["lambda"].item()
        if regime == "Full":
            final_models[(regime, regularizer)] = full_grid_models[(regularizer, lam)]
        else:
            final_models[(regime, regularizer)] = fit_model(X, y, **penalty(regularizer, lam))

print(f"{len(full_grid_models)} Full-data grid models and {len(final_models)} final models trained")

In [ ]:
# Weight images of the baseline and of the CV-selected models, side by side.
# Each row has its own symmetric color scale, since the penalties change the size of the
# weights a lot: the shape of each row is what is compared here, and the sizes are in
# the table below

def model_label(regime: str, regularizer: str) -> str:
    if regularizer == "None":
        return "No regularization"
    lam = selected.query("regime == @regime and regularizer == @regularizer")["lambda"].item()
    return f"{regularizer}, $\\lambda$={lam:g}"


def plot_weight_comparison(regime: str) -> None:
    regularizers = ["None", *LAMBDA_GRID]
    fig, axs = plt.subplots(len(regularizers), 10, figsize=(17, 2 * len(regularizers) + 0.6))

    for row, regularizer in enumerate(regularizers):
        weights = get_weights(final_models[(regime, regularizer)])
        limit = np.abs(weights).max()

        for digit in range(10):
            ax = axs[row, digit]
            im = ax.imshow(weights[digit].reshape(28, 28), cmap="icefire", vmin=-limit, vmax=limit)
            ax.set_xticks([])
            ax.set_yticks([])
            if row == 0:
                ax.set_title(f"digit {digit}", fontsize=9)

        axs[row, 0].set_ylabel(model_label(regime, regularizer), fontsize=9)
        fig.colorbar(im, ax=axs[row, :].tolist(), shrink=0.9, pad=0.01)

    plt.suptitle(f"Weight images, {regime}-data", fontsize=13)
    plt.show()


for regime in data_regimes:
    plot_weight_comparison(regime)

weights_table = pd.DataFrame([
    {"regime": regime, "model": model_label(regime, regularizer).replace("$\\lambda$", "lambda"),
     **weight_summary(get_weights(fitted))}
    for (regime, regularizer), fitted in final_models.items()
])
display(weights_table.round(3))

> **Question 3**
>
> (a) As $\lambda$ increases for L2 regularization, how does CV accuracy change on each data regime?
> Are there clear optima, or does accuracy plateau? Is the difference between the best $\lambda$ and its neighbours larger than the fold-to-fold standard deviation?
>
> (b) Compare the weight images of the CV-selected L2 model, the CV-selected L1 model, and the unregularized baseline. What differences do you observe in terms of structure and sparsity?
>
> (c) Connect the weight images to the corruption structure identified in Part 1. Does the CV-selected model appear to have suppressed the corruption pattern, amplified it, or is it ambiguous from weights alone?
>
> (d) **Predict**, based on what you see in the weights and CV curves, which of your three CV-selected models (none / L2 / L1) you expect will perform best on clean images, and which on images with impulse noise. Justify your prediction before seeing any test results.

**(a)**

A tabela e o gráfico **“5-fold CV accuracy on the corrupted training data”** mostram que, com L2, valores pequenos de $\lambda$ praticamente reproduzem o baseline. Com o aumento da penalização, há uma leve melhora no Scarce e no Partial seguida de queda, enquanto no regime Full, a melhora é praticamente inexistente. Em $\lambda=1$, as acurácias caem para $0{,}7140$, $0{,}7587$ e $0{,}7533$, respectivamente. A queda começa por volta de $\lambda = 0{,}1$ no Scarce, $\lambda = 0{,}03$ no Partial e $\lambda = 0{,}001$ no Full.

Assim, a regularização passa a prejudicar o desempenho em valores menores de $\lambda$ à medida que aumenta a quantidade de dados. Esse comportamento é compatível com o compromisso entre viés e variância: com poucos exemplos, a penalização pode reduzir o ajuste a particularidades da amostra, compensando o viés introduzido. A maior norma dos pesos e a maior participação das bordas no Scarce, observadas na Questão 2.3, são compatíveis com essa interpretação, mas não constituem medidas diretas da variância. Com mais dados, o benefício da regularização tende a ser menor, e penalizações mais fortes podem restringir padrões úteis à classificação, reduzindo a acurácia.

Não há ótimos bem definidos, mas sim platôs. Os ganhos do melhor $\lambda$ sobre o baseline são pequenos: $0{,}0060$ no Scarce, $0{,}0050$ no Partial e $0{,}0001$ no Full. No Scarce, $\lambda = 0{,}003$ e $\lambda = 0{,}03$ empatam $(0{,}8290)$, e a escolha do primeiro se deve apenas ao critério de desempate, que favorece o menor $\lambda$.

As diferenças entre o melhor $\lambda$ e seus vizinhos são menores que o desvio padrão entre folds em todos os regimes. Como o melhor $\lambda$ foi escolhido justamente por ter a maior média, essas comparações já o favorecem. Portanto, o $\lambda$ selecionado não é distinguível de seus vizinhos.

**(b)**

Os três modelos preservam estruturas centrais associadas aos dígitos, mas diferem na magnitude e na distribuição espacial dos pesos. A regularização L2 reduz a magnitude dos pesos sem aumentar substancialmente a esparsidade. No Scarce, a norma diminui de $13{,}653$ no baseline para $9{,}015$; no Partial, de $11{,}929$ para $7{,}986$; e no Full, de $9{,}798$ para $9{,}357$. A redução é, portanto, mais acentuada nos regimes com menos dados.

Já L1 produz mapas com mais regiões de pesos nulos ou próximos de zero, sobretudo no fundo e nas bordas, preservando parte das estruturas discriminativas centrais. Considerando a esparsidade como a fração de pesos com $|w|<10^{-3}$, ela aumenta de $0{,}8\%$ para $54{,}1\%$ no Scarce, de $0{,}7\%$ para $58{,}1\%$ no Partial e de $1{,}4\%$ para $9{,}6\%$ no Full. Com L2, essa fração permanece entre $0{,}8\%$ e $1{,}5\%$. No Full, as diferenças visuais entre os três modelos são menores, em concordância com a seleção de penalizações mais fracas pela validação cruzada.

**(c)**

Na Parte 1, identificamos uma corrupção formada por traços em zigue-zague, cuja posição e orientação variam entre imagens e que pode se sobrepor aos dígitos. Como esses traços não carregam informação sobre o dígito, um modelo robusto deveria atribuir pesos próximos de zero aos pixels em que praticamente só a corrupção aparece. Esses pixels se concentram na faixa de borda, que é dominada pela corrupção.

Os modelos com L1 se aproximam do comportamento idealizado, pois além de zerarem muitos pesos do fundo, reduzem a participação da borda na magnitude dos pesos de $25{,}8\%$ para $15{,}2\%$ no Scarce e de $24{,}3\%$ para $7{,}6\%$ no Partial, o que sugere a supressão da corrupção.

Os modelos L2, por outro lado, reduzem os pesos de forma aproximadamente proporcional, e a participação da borda muda pouco no Scarce (de $25{,}8\%$ para $25{,}3\%$) e moderadamente no Partial (de $24{,}3\%$ para $20{,}7\%$), de modo que a influência relativa da corrupção nessa região é apenas parcialmente reduzida. No Full, a participação da borda já é menor no baseline ($17{,}8\%$) e varia pouco com a regularização ($17{,}3\%$ com L2 e $14{,}7\%$ com L1).

Na região central, porém, a interpretação é ambígua. Como o modelo é linear, ele pondera cada pixel de forma independente e não reconhece o formato de um traço: só consegue ignorar a corrupção reduzindo os pesos dos pixels em que ela aparece. No centro, esses mesmos pixels também contêm o dígito, e, como a posição da corrupção muda a cada imagem, ela não forma um padrão identificável nos pesos. Nenhum mapa apresenta estruturas em forma de zigue-zague, o que não indica amplificação, mas os pesos, por si só, não mostram como o modelo reage a um traço que cruza o dígito. Assim, os pesos sugerem que o L1 suprime a corrupção na periferia, mas o efeito no centro precisa ser avaliado pelo desempenho nos conjuntos de teste.

**(d)**

Esperamos que o modelo selecionado com L1 tenha o melhor desempenho no conjunto com ruído impulsivo, seguido por L2 e pelo baseline. Nossa hipótese é que a maior esparsidade e a menor participação das bordas nos pesos reduzam a influência de parte dos pixels perturbados, preservando estruturas centrais úteis à classificação. L2 também pode ajudar pela redução das magnitudes dos pesos. Entretanto, essa ordenação é incerta, pois o ruído impulsivo também pode atingir pixels centrais com pesos elevados no modelo L1.

Nas imagens limpas, também prevemos uma leve vantagem de L1, pela possível redução da dependência de associações ocasionais presentes no treinamento corrompido. A ausência da corrupção pode facilitar o reconhecimento, mas não garante acurácia superior à obtida nas imagens corrompidas, pois há mudança de distribuição. No Full, esperamos diferenças menores entre os três modelos, considerando a semelhança dos mapas e as penalizações mais fracas selecionadas. Essas previsões permanecem hipóteses, já que a validação cruzada avalia apenas a corrupção utilizada no treinamento.

---

## Part 4. Test

**(2.0 points)**

Until now, all decisions, dataset size analysis, regularization selection, were made using only the corrupted training data and cross-validation.
**Now we use the test sets** to evaluate the models you selected.

1. Evaluate all the chosen models (baseline, CV-selected L2 and L1, for each data regime) on the three test splits.
2. Using the Full-data models stored for every $\lambda$, plot the **trade-off curves**: accuracy on `test_corrupted` versus accuracy on `test_clean`, and versus `test_impulse`, one curve per regularizer, marking the CV-selected models.
3. Plot the **sparsity** (fraction of weights with $|w| < 10^{-3}$) of the L1 and L2 models against $\lambda$.
4. Compute the **per-class accuracy** on each test split for the CV-selected models and identify which digits suffer the most from each corruption.

In [ ]:
# Load the test datasets and evaluate all the chosen models for each data regime

y_test = np.load(f"{DATA_DIR}/test/test_labels.npy")
test_splits = {
    "corrupted": np.load(f"{DATA_DIR}/test/test_images.npy"),
    "clean": np.load(f"{DATA_DIR}/test/clean_images.npy"),
    "impulse": np.load(f"{DATA_DIR}/test/impulse_images.npy"),
}

for split, X in test_splits.items():
    print(f"test_{split}: {X.shape}, {X.dtype}")

# The three splits hold the same 10000 digits, only the perturbation changes
n_examples = 8
example_idxs = np.random.default_rng(seed).choice(len(y_test), size=n_examples, replace=False)

fig, axs = plt.subplots(len(test_splits), n_examples, figsize=(1.8 * n_examples, 6))
for row, (split, X) in enumerate(test_splits.items()):
    for col, idx in enumerate(example_idxs):
        axs[row, col].imshow(X[idx], cmap="gray", vmin=0, vmax=255)
        axs[row, col].set_xticks([])
        axs[row, col].set_yticks([])
        if row == 0:
            axs[row, col].set_title(f"target {y_test[idx]}", fontsize=9)
    axs[row, 0].set_ylabel(f"test_{split}", fontsize=10)

plt.suptitle("The same test digits in the three splits", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 1. Accuracy of every chosen model on the three test splits

def test_accuracies(fitted: FittedModel) -> dict[str, float]:
    accs = {f"acc {split}": evaluate(fitted, X, y_test) for split, X in test_splits.items()}
    accs["gap clean"] = accs["acc corrupted"] - accs["acc clean"]
    accs["gap impulse"] = accs["acc corrupted"] - accs["acc impulse"]
    return accs


test_results = pd.DataFrame([
    {
        "regime": regime,
        "regularizer": regularizer,
        "lambda": 0.0 if regularizer == "None"
        else selected.query("regime == @regime and regularizer == @regularizer")["lambda"].item(),
        "cv_mean": cv_results.query(
            "regime == @regime and regularizer == @regularizer"
        ).set_index("lambda")["cv_mean"].get(
            0.0 if regularizer == "None"
            else selected.query("regime == @regime and regularizer == @regularizer")["lambda"].item()
        ),
        **test_accuracies(fitted),
    }
    for (regime, regularizer), fitted in final_models.items()
])

display(test_results.astype({"lambda": str}).round(4))

# Same numbers as grouped bars: one panel per split, one group per regime
fig, axs = plt.subplots(1, len(test_splits), figsize=(18, 4.5), sharey=True, constrained_layout=True)
bar_width = 0.27
regimes = list(data_regimes)

for ax, split in zip(axs, test_splits):
    for i, regularizer in enumerate(["None", *LAMBDA_GRID]):
        values = [
            test_results.query("regime == @regime and regularizer == @regularizer")[f"acc {split}"].item()
            for regime in regimes
        ]
        positions = np.arange(len(regimes)) + (i - 1) * bar_width
        bars = ax.bar(positions, values, width=bar_width, color=reg_colors[regularizer],
                      label="No regularization" if regularizer == "None" else f"{regularizer} (CV-selected)")
        ax.bar_label(bars, fmt="%.3f", fontsize=7, padding=2)

    ax.set_xticks(np.arange(len(regimes)))
    ax.set_xticklabels([f"{regime}-data" for regime in regimes])
    ax.set_title(f"test_{split}")
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.3)

axs[0].set_ylabel("Accuracy")
axs[0].legend(frameon=False, loc="lower left", fontsize=8)
plt.suptitle("Test accuracy of the chosen models", fontsize=13)
plt.show()

In [ ]:
# 2. Trade-off curves on the Full-data regime: every lambda of the grid evaluated on the
# three test splits. Each curve starts at the smallest lambda, next to the unregularized
# baseline (gray square), and the stars mark the CV-selected models. Like the grid models
# themselves, these numbers are only a diagnostic: nothing is selected with them

grid_tests = pd.DataFrame([
    {"regularizer": regularizer, "lambda": lam, **test_accuracies(fitted)}
    for (regularizer, lam), fitted in [(("None", 0.0), final_models[("Full", "None")]), *full_grid_models.items()]
])
full_cv = cv_results.query("regime == 'Full'")[["regularizer", "lambda", "cv_mean", "cv_std"]]
grid_results = full_cv.merge(grid_tests, on=["regularizer", "lambda"])

# The large lambdas pull the axes far from the baseline, so the bottom row zooms in on
# the models within one accuracy point of it on test_corrupted, where the CV-selected
# models are and where the two accuracies can improve together
ZOOM_WINDOW = 0.01


def plot_tradeoff(ax: plt.Axes, split: str, data: pd.DataFrame) -> None:
    baseline_row = data.query("regularizer == 'None'").iloc[0]
    ax.scatter(baseline_row["acc corrupted"], baseline_row[f"acc {split}"], marker="s", s=80,
               color=reg_colors["None"], edgecolor="black", zorder=5, label="No regularization")

    for regularizer in LAMBDA_GRID:
        curve = data.query("regularizer == @regularizer").sort_values("lambda")
        color = reg_colors[regularizer]
        ax.plot(curve["acc corrupted"], curve[f"acc {split}"], marker="o", color=color, label=regularizer)
        for _, row in curve.iterrows():
            ax.annotate(f"{row['lambda']:g}", (row["acc corrupted"], row[f"acc {split}"]),
                        textcoords="offset points", xytext=(5, 5), fontsize=7, color=color)

        lam = selected.query("regime == 'Full' and regularizer == @regularizer")["lambda"].item()
        choice = curve[curve["lambda"] == lam].iloc[0]
        ax.scatter(choice["acc corrupted"], choice[f"acc {split}"], marker="*", s=250, color=color,
                   edgecolor="black", alpha=0.8, zorder=6)

    ax.set_xlabel("Accuracy on test_corrupted")
    ax.set_ylabel(f"Accuracy on test_{split}")
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, loc="best")


baseline_acc = grid_results.query("regularizer == 'None'")["acc corrupted"].item()
near_baseline = grid_results[grid_results["acc corrupted"] >= baseline_acc - ZOOM_WINDOW]

fig, axs = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True)

for col, split in enumerate(["clean", "impulse"]):
    plot_tradeoff(axs[0, col], split, grid_results)
    axs[0, col].set_title(f"test_corrupted vs test_{split}, every lambda")
    plot_tradeoff(axs[1, col], split, near_baseline)
    axs[1, col].set_title(f"test_corrupted vs test_{split}, zoom near the baseline (0.001 = {len(y_test) // 1000} test images)")

plt.suptitle("Trade-off curves of the Full-data models (labels: lambda; stars: CV-selected)", fontsize=13)
plt.show()

display(grid_results.astype({"lambda": str}).round(4))

In [ ]:
# 3. Sparsity of the Full-data models against lambda: the fraction of weights with
# |w| < SPARSITY_THRESHOLD. The L1 proximal step of train can set weights exactly to
# zero, while L2 only shrinks them towards zero, so the two curves can differ a lot

sparsity_results = pd.DataFrame([
    {"regularizer": regularizer, "lambda": lam, "sparsity": sparsity(get_weights(fitted))}
    for (regularizer, lam), fitted in full_grid_models.items()
])
baseline_sparsity = sparsity(get_weights(final_models[("Full", "None")]))

fig, ax = plt.subplots(figsize=(10, 5))
ax.axhline(baseline_sparsity, color=reg_colors["None"], linestyle="--", label="No regularization")

for regularizer in LAMBDA_GRID:
    curve = sparsity_results.query("regularizer == @regularizer")
    color = reg_colors[regularizer]
    ax.plot(curve["lambda"], curve["sparsity"], marker="o", color=color, label=regularizer)

    lam = selected.query("regime == 'Full' and regularizer == @regularizer")["lambda"].item()
    ax.scatter(lam, curve.loc[curve["lambda"] == lam, "sparsity"].item(), marker="*", s=250,
               color=color, edgecolor="black", zorder=5)

ax.set_xscale("log")
ax.set_ylim(0, 1)
ax.set_xlabel(r"$\lambda$")
ax.set_ylabel(f"Fraction of weights with |w| < {SPARSITY_THRESHOLD:g}")
ax.set_title("Sparsity of the Full-data models (stars: CV-selected)")
ax.grid(alpha=0.3)
ax.legend(frameon=False)
plt.show()

display(sparsity_results.pivot(index="lambda", columns="regularizer", values="sparsity").round(3))

In [ ]:
# 4. Per-class accuracy of the chosen models on each test split. The three splits hold
# the same digits, so the accuracy lost from test_clean to a corrupted split measures
# how much each digit suffers from that corruption alone

def per_class_accuracy(fitted: FittedModel, X: NDArray[np.uint8]) -> NDArray[np.float64]:
    """Accuracy on the test images of each digit (the recall of each class)."""
    y_pred = predict_logits(fitted, X).argmax(dim=-1).numpy()
    return np.array([(y_pred[y_test == digit] == digit).mean() for digit in range(N_CLASSES)])


per_class = {
    (regime, regularizer, split): per_class_accuracy(fitted, X)
    for (regime, regularizer), fitted in final_models.items()
    for split, X in test_splits.items()
}

# Top row: per-class accuracy on each split. Bottom row: accuracy lost on each digit
# relative to test_clean. One column per chosen Full-data model
split_colors = {"corrupted": "tab:red", "clean": "tab:green", "impulse": "tab:purple"}
digits = np.arange(N_CLASSES)

fig, axs = plt.subplots(2, 3, figsize=(18, 9), sharey="row", constrained_layout=True)

for col, regularizer in enumerate(["None", *LAMBDA_GRID]):
    for i, (split, color) in enumerate(split_colors.items()):
        axs[0, col].scatter(digits + (i - 1) * 0.2, per_class[("Full", regularizer, split)], color=color,
                            label=f"test_{split}")

    for i, split in enumerate(["corrupted", "impulse"]):
        drop = per_class[("Full", regularizer, "clean")] - per_class[("Full", regularizer, split)]
        axs[1, col].bar(digits + (i - 0.5) * 0.4, drop, width=0.4, color=split_colors[split],
                        label=f"test_{split}")

    axs[0, col].set_title(model_label("Full", regularizer))
    axs[1, col].axhline(0, color="black", linewidth=0.8)
    for ax in axs[:, col]:
        ax.set_xticks(digits)
        ax.set_xlabel("Digit")
        ax.grid(alpha=0.3)

axs[0, 0].set_ylabel("Per-class accuracy")
axs[1, 0].set_ylabel("Accuracy lost relative to test_clean")
axs[0, 0].legend(frameon=False)
axs[1, 0].legend(frameon=False)
plt.suptitle("Per-class test accuracy of the chosen Full-data models", fontsize=13)
plt.show()

# Accuracy lost on each digit relative to test_clean, for every chosen model, with the
# three digits that lose the most
drops = pd.DataFrame([
    {"regime": regime, "regularizer": regularizer, "corruption": split,
     **dict(zip(digits, per_class[(regime, regularizer, "clean")] - per_class[(regime, regularizer, split)]))}
    for (regime, regularizer) in final_models
    for split in ["corrupted", "impulse"]
]).set_index(["regime", "regularizer", "corruption"])
drops["most affected"] = drops[digits].apply(lambda row: ", ".join(map(str, row.nlargest(3).index)), axis=1)

display(drops.round(3))

> **Question 4**
>
> (a) Describe the shape of the trade-off curve (corrupted-test vs. clean-test). Is there a region where increasing regularization simultaneously improves both accuracies? Is there a region of genuine trade-off?
>
> (b) The CV-selected models are marked on the plot. Where do they fall relative to the trade-off frontier? What does this tell you about using cross-validation as a model selection criterion when training and deployment distributions differ?
>
> (c) Revisit your prediction from Question 3(d). Was it correct? If not, what did the weight images fail to reveal that the test results exposed?
>
> (d) Connect the sparsity plot to the out-of-distribution results. Why might L1 regularization behave differently from L2 when evaluated on clean images and on impulse noise?
>
> (e) Compare `test_clean` and `test_impulse`. Did training on one corruption make the model robust to a different one? Which data regime and which regularizer transferred best, and why?
>
> (f) Use the per-class accuracies to check your answer to Question 1.1(c). Which digits suffer the most under each corruption?
>
> (g) In a real deployment scenario, neither the clean nor the impulse test set would be available during model selection. Given only corrupted validation data, what practical strategies could a practitioner use to make a better model selection decision than simply maximizing CV accuracy?

**(a)**

As curvas do Full evidenciam uma faixa restrita de regularização, próxima ao *baseline*, na qual ambas as acurácias apresentam melhorias simultâneas, como observado em L2 para $\lambda=10^{-4}$ e em L1 para $\lambda=3\times10^{-5}$. Nessa faixa, entretanto, também são identificados *trade-offs* locais entre os desempenhos nos dois conjuntos de teste. Em L1, o aumento de $\lambda$ de $3\times10^{-5}$ para $10^{-4}$ eleva a acurácia em imagens limpas de $0{,}9066$ para $0{,}9071$, ao mesmo tempo que reduz a acurácia em imagens corrompidas de $0{,}8929$ para $0{,}8920$. Em L2, a variação de $\lambda$ de $10^{-4}$ para $10^{-3}$ produz o comportamento oposto, com aumento da acurácia em imagens corrompidas e redução daquela obtida em imagens limpas. Essas variações, porém, são muito pequenas e correspondem a poucas imagens entre as 10.000 do conjunto de teste, não indicando relevância prática.

Para valores mais elevados de $\lambda$, ambas as curvas passam a seguir trajetórias aproximadamente lineares e diagonais, caracterizadas pela redução simultânea das duas acurácias e pelo deslocamento em direção à região inferior esquerda do gráfico. Dessa forma, os *trade-offs* ficam restritos a uma faixa estreita de regularização, enquanto penalizações mais intensas provocam uma deterioração conjunta do desempenho.

Observa-se, ainda, que a acurácia em imagens limpas permanece superior à obtida em imagens corrompidas ao longo de ambas as curvas, com a curva de L1 situada acima da de L2. A comparação entre configurações com acurácias semelhantes em imagens corrompidas, próximas de $0{,}854$, evidencia uma vantagem de L1 na preservação do desempenho em imagens limpas, com acurácia de $0{,}8705$, frente a $0{,}8621$ em L2. Esse resultado indica que, nessa região, a regularização L1 proporciona um equilíbrio mais favorável entre o desempenho em imagens limpas e a robustez às corrupções.

**(b)**

Os modelos selecionados por CV situam-se próximos à região de maior acurácia em imagens corrompidas. Considerando conjuntamente L1 e L2, o modelo L1 selecionado ($\lambda=3\times10^{-5}$) integra a fronteira de trade-off observada em ambos os gráficos, enquanto o L2 selecionado ($\lambda=3\times10^{-4}$) é ligeiramente dominado por ele. Embora ambos alcancem $0{,}8929$ em imagens corrompidas, L1 apresenta acurácias ligeiramente maiores nas limpas e nas impulsivas. Entretanto, o L1 selecionado não maximiza o desempenho nas imagens impulsivas, visto que com $\lambda=10^{-4}$ a acurácia sobe de $0{,}8862$ para $0{,}8902$, enquanto nas corrompidas cai apenas de $0{,}8929$ para $0{,}8920$.

Esses resultados evidenciam que a seleção por validação cruzada na distribuição de treinamento não garante o melhor desempenho sob mudanças de distribuição. Ao privilegiar a acurácia na distribuição representada durante a validação, a CV pode selecionar configurações que não oferecem o compromisso mais favorável entre as diferentes condições de teste. Nesse contexto, outras intensidades de regularização podem proporcionar um equilíbrio mais adequado entre o desempenho na corrupção conhecida e a capacidade de generalização para distribuições distintas.

**(c)**

A previsão foi parcialmente confirmada. Sob ruído impulsivo, ambos os modelos regularizados superaram o baseline nos três regimes, mantendo-se a ordenação L1 > L2 > baseline. Entretanto, a vantagem de L1 sobre L2 foi mais expressiva no Partial ($0{,}8841$ contra $0{,}8776$), enquanto no Scarce e no Full correspondeu a apenas seis e oito imagens corretamente classificadas a mais, respectivamente. Nas imagens limpas, L1 também apresentou o melhor desempenho no Partial ($0{,}9020$, frente a $0{,}8985$ do baseline e $0{,}8980$ de L2), ao passo que, no Full, os três modelos alcançaram acurácias praticamente equivalentes, conforme previsto. No Scarce, contudo, L1 apresentou um ganho desprezível em relação ao baseline ($0{,}8666$ contra $0{,}8664$), enquanto L2 obteve a maior acurácia ($0{,}8686$).

Esses resultados indicam que o aumento da esparsidade e a redução da contribuição das bordas não se traduzem necessariamente em melhorias na classificação. Embora os mapas permitam identificar quais pesos foram reduzidos ou eliminados pela regularização, eles não revelam se esses pesos eram relevantes para a discriminação de amostras específicas, assim como não permitem quantificar diretamente o impacto dessas alterações nas decisões do modelo. Os resultados de acurácia, por sua vez, mostram que os efeitos da regularização sobre o desempenho foram substancialmente mais pronunciados no Partial do que nos demais regimes.

**(d)**

O gráfico mostra que L1 aumenta rapidamente a esparsidade com $\lambda$, enquanto L2 atua principalmente reduzindo a magnitude dos pesos. Ao zerar pesos, L1 pode diminuir a dependência de associações incorretas e a influência de pixels atingidos pelo ruído impulsivo. Porém, também pode eliminar informações úteis dos dígitos, e os pixels que mantêm pesos elevados continuam vulneráveis ao ruído. L2 preserva contribuições distribuídas por mais pixels, embora com menor intensidade, de modo que cada pixel ruidoso continua contribuindo para a decisão.

No Full, o ganho de L1 no ruído impulsivo surge quando a esparsidade passa de $9{,}6\%$, no $\lambda$ selecionado, para valores entre $36{,}5\%$ e $64{,}9\%$, faixa em que L1 supera o melhor resultado obtido com L2, que praticamente não gera esparsidade enquanto mantém acurácia próxima à do baseline. Nas imagens limpas, as variações nessa faixa são pequenas, e, quando a esparsidade atinge $80{,}0\%$, as acurácias caem em todos os conjuntos. Portanto, a seleção de pesos promovida por L1 pode oferecer vantagens diferentes da redução de magnitudes promovida por L2, mas aumentar a esparsidade não garante maior robustez.

**(e)**

Em todos os modelos, a acurácia em test_impulse é inferior à de test_clean, com diferenças entre $0{,}0179$ e $0{,}0334$, mas permanece próxima à de test_corrupted, ficando, no máximo, $0{,}0121$ abaixo dela. Os resultados indicam tolerância parcial ao ruído impulsivo, embora as perdas variem entre as classes, como observado para o dígito 1 em (f).

O L1 no Partial destaca-se pelo maior ganho sobre o baseline em test_impulse, de $0{,}8651$ para $0{,}8841$, e pela menor diferença em relação a test_clean, de $0{,}0179$. Sua acurácia no ruído impulsivo também supera a obtida em test_corrupted ($0{,}8782$). Esse comportamento é compatível com a menor participação das bordas na magnitude dos pesos ($7{,}6\%$), que pode reduzir a influência de perturbações periféricas. Em termos absolutos, entretanto, o melhor resultado é o do L1 no Full ($0{,}8862$), com uma melhora pequena sobre o baseline ($0{,}8843$).


**(f)**

Considerando a redução de acurácia em relação às imagens limpas, observa-se que, no Full, os dígitos 5 e 4 apresentam as maiores perdas sob a corrupção zigue-zague, enquanto 1, 5 e 4 são os mais afetados pelo ruído impulsivo. No Scarce, as maiores reduções ocorrem nos dígitos 3 e 4 para o zigue-zague e 4 e 1 para o ruído impulsivo. Já no Partial, os dígitos 5 e 4 são os mais prejudicados pelo zigue-zague, enquanto 1, 5 e 7 concentram as maiores perdas sob ruído impulsivo.

Esses resultados confirmam apenas parcialmente a hipótese formulada na Questão 1.1(c), que apontava os dígitos 1, 7 e 4 como particularmente vulneráveis. O dígito 4 figura entre os mais afetados pelo zigue-zague nos três regimes, e o 7 aparece entre eles no Partial, mas o 1 praticamente não se destaca nessa corrupção (perda de $0{,}017$ no Full, semelhante à da maioria dos dígitos), sendo, em vez disso, o mais prejudicado pelo ruído impulsivo ($0{,}086$ no Full). Além disso, a vulnerabilidade recorrente do dígito 5, especialmente ao zigue-zague, evidencia um comportamento não antecipado pela hipótese inicial.

**(g)**

Uma estratégia seria considerar a estabilidade do desempenho em validações cruzadas repetidas e, entre configurações com acurácias semelhantes, preferir uma regularização moderada que reduza a sensibilidade a perturbações. Também poderíamos criar cenários de validação com alterações nas imagens disponíveis, que reflitam condições esperadas de uso do modelo após deploy e enriqueçam nosso conjunto de treinamento/validação, avaliando tanto a acurácia média quanto o pior desempenho entre cenários para verificarmos o comportamento diante de perturbações encontradas em cenários reais. Assim, seria possível avaliarmos aspectos de robustez que a validação com uma única corrupção não revela.

---

## Summary

Complete the table below with results from your experiments (Full-data regime). Use the best model according to each criterion.

| Model                              | $\lambda$ | CV Accuracy (mean ± std) | Test Acc (corrupted) | Test Acc (clean) | Test Acc (impulse) | Gap clean | Gap impulse |
|------------------------------------|-----------|--------------------------|----------------------|------------------|--------------------|-----------|-------------|
| No regularization (baseline)       | 0         | 0.8842 ± 0.0022          | 0.8925               | 0.9063           | 0.8843             | -0.0138   | 0.0082      |
| Best CV, L2                        | 3e-4      | 0.8843 ± 0.0019          | 0.8929               | 0.9063           | 0.8854             | -0.0134   | 0.0075      |
| Best CV, L1                        | 3e-5      | 0.8846 ± 0.0018          | 0.8929               | 0.9066           | 0.8862             | -0.0137   | 0.0067      |
| Smallest clean gap, L2             | 1         | 0.7533 ± 0.0029          | 0.7681               | 0.7700           | 0.7821             | -0.0019   | -0.0140     |
| Smallest clean gap, L1             | 1e-5      | 0.8843 ± 0.0021          | 0.8929               | 0.9064           | 0.8850             | -0.0135   | 0.0079      |
| Smallest impulse gap, L2           | 1e-2      | 0.8758 ± 0.0022          | 0.8851               | 0.8957           | 0.8843             | -0.0106   | 0.0008      |
| Smallest impulse gap, L1           | 1e-3      | 0.8711 ± 0.0020          | 0.8795               | 0.8961           | 0.8800             | -0.0166   | -0.0005     |

*Gap clean = Test Acc (corrupted) - Test Acc (clean); Gap impulse = Test Acc (corrupted) - Test Acc (impulse).*

*Smallest gap: gap closest to zero (smallest |gap|).*

In [ ]:
# Numbers of the Summary table above, taken from grid_results (Full-data regime).
# "Smallest gap" is read as the gap closest to zero: the model whose accuracy changes
# the least between test_corrupted and the other split

summary_rows = {"No regularization (baseline)": grid_results.query("regularizer == 'None'").iloc[0]}

for regularizer in LAMBDA_GRID:
    curve = grid_results.query("regularizer == @regularizer")
    lam = selected.query("regime == 'Full' and regularizer == @regularizer")["lambda"].item()
    summary_rows[f"Best CV, {regularizer}"] = curve[curve["lambda"] == lam].iloc[0]

for gap in ["clean", "impulse"]:
    for regularizer in LAMBDA_GRID:
        curve = grid_results.query("regularizer == @regularizer")
        summary_rows[f"Smallest {gap} gap, {regularizer}"] = curve.loc[curve[f"gap {gap}"].abs().idxmin()]

summary = pd.DataFrame(summary_rows).T.drop(columns="regularizer").astype(float)
summary.index.name = "model"

display(summary.astype({"lambda": str}).round(4))

---
### **Declaração de uso de IA generativa**

Conforme a diretriz 4, declaramos que utilizamos o modelo Claude Opus 5.5 (Anthropic), por meio da extensão Claude Code para o VS Code, como assistente de implementação durante a realização desta tarefa.

Os prompts empregados foram, em sua maioria, dos seguintes tipos:

- otimização do tempo de execução (por exemplo, como otimizar o tempo do fit na validação cruzada, o que levou à função `fit_models`, que treina os modelos da grade de $\lambda$ simultaneamente);
- correção de bugs e de problemas de exibição (por exemplo, o $\lambda$ = 3e-05 arredondado para 0.0000 nas tabelas);
- melhoria da legibilidade dos gráficos e da organização do código;
- verificação de consistência entre os números da tabela do Summary e as saídas do notebook;
- correção gramatical pontual nas respostas.

As respostas às questões foram redigidas pelos integrantes do grupo, e todo o código gerado foi revisado por eles antes de ser incorporado.

---

## Assignment Submission

This notebook must be filled with your solution and submitted on the Assignment's entry at our Google Classroom page.

Before submitting, verify that:
- The notebook runs end-to-end without errors (use **Run All** after restarting the kernel).
- All plots are rendered in the output.
- All written answers are filled in.
- The summary table is completed.
- AI usage declaration and prompts are included if applicable.

**Important: Only one member of the group should submit.**

### Policy for late submissions

You are NOT encouraged to submit after the deadline; however, in this case the following penalties apply:

- 25% deduction for 1-day late submission.
- 50% deduction for 2-day late submission.
- 75% deduction for 3-day late submission.